# CBB Analytics REST API — sample Python client

Docs: https://rest.cbbanalytics.com/api-docs/#/

**Major versions** (response shape is decided by URL prefix — no envelope query param):

| | Fields | Response shape |
|---|---|---|
| **v3** recommended | All current fields, evolving | `{ response: { meta, data } }` envelope; `meta` carries `count`, `limit`, `offset`, `nextCursor`, `hasMore`, `version`, `time`, and the echoed request URL (apikey stripped) |
| **v2** legacy | All current fields, evolving (same as v3) | Bare JSON array |
| **v1** legacy | Frozen Zod-picked field set | Bare JSON array |

**Cursor pagination** — pass `after=<cursor>` to fetch rows with `_id > after`. On v1/v2 the cursor is in the `X-Next-Cursor` response header; on v3 it's also exposed as `meta.nextCursor` in the body. Stop when the cursor is absent (or `meta.hasMore` is `False`). Only compatible with the default sort (omit `sortBy`).

**What this notebook gives you**:
- `api_get` / `paging_offset` / `paging_cursor` / `fetch_data` — request helpers that auto-detect envelope vs bare-array, so the same code works on v1, v2, v3.
- Realistic backfill examples in section (D) — entities, team/player stats, and incremental syncs via the `updated` filter.
- `test_versions(table, params)` — same query across versions; field-set diff.
- `bench_pagination(table, params)` — same query across (version × mode); timings + headline cursor-vs-offset comparison.
- Section (E) below is a runnable test menu using those helpers.

##
## (A) Setup

In [ ]:
# check if libraries already downloaded
import importlib.util
import sys
required = ['pandas', 'requests']
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
if missing:
    print(f'Installing missing packages: {", ".join(missing)}')
    # subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing]) # uncomment to install missing libraries
    print('Done — you may need to restart the kernel before re-running.')
else:
    print(f'All required libraries already installed: {", ".join(required)}')

In [ ]:
import sys
import time
import re
from typing import Any, Iterable, Optional
from datetime import date, timedelta
import pandas as pd
import requests
print(f'python {sys.version.split()[0]}  pandas {pd.__version__}  requests {requests.__version__}')

# Globals — edit these for your environment.
API_KEY = 'YOUR_API_KEY_HERE'
HOST = 'https://rest.cbbanalytics.com'

# 'v3' = envelope + all current fields (recommended)
# 'v2' = bare array + all current fields (legacy)
# 'v1' = bare array + frozen Zod-picked schema (legacy)
API_VER = 'v3' # v1, v2, v3
BASE_URL = f'{HOST}/{API_VER}'
PAGING_MODE = 'cursor' # offset, cursor
LATEST_COMPETITION_ID = 41097
ACC_CONFERENCE_ID = 53

##
## (B) Low-level request helpers

In [ ]:
def api_get(url: str, query_params: dict, api_key: str = API_KEY, timeout: int = 60) -> dict:
    """Single GET. Returns {records, headers, meta, latency_ms}.

    Auto-detects envelope vs bare-array responses, so the same code works
    against /v1/, /v2/ (bare array) and /v3/ (envelope). No flag needed —
    the server's response shape is determined by the major URL version.
    """
    t0 = time.perf_counter()
    res = requests.get(
        url,
        headers={'X-API-Key': api_key},
        params=query_params,
        timeout=timeout,
    )
    latency_ms = (time.perf_counter() - t0) * 1000.0

    if res.status_code != 200:
        raise RuntimeError(f'API request failed: {res.status_code} {res.text}')

    body = res.json()

    # Auto-detect envelope (v3) vs bare array (v1/v2). v3 wraps in
    # { response: { meta, data } }; v1 and v2 return the array directly.
    is_envelope = (
        isinstance(body, dict)
        and isinstance(body.get('response'), dict)
        and 'data' in body['response']
    )
    if is_envelope:
        records = body['response']['data']
        meta    = body['response']['meta']
    else:
        records = body
        meta    = None

    return {
        'records':    records,
        'headers':    res.headers,  # case-insensitive in `requests`
        'meta':       meta,
        'latency_ms': latency_ms,
    }

In [ ]:
def paging_offset(url: str, query_params: dict, quiet: bool = False) -> dict:
    """Offset-based paging. Returns {records, timings}.

    How it works:
      - Pass `offset` to skip the first `offset` records and fetch the next `limit`.
      - Increment `offset` by `limit` for each subsequent page.
      - Paging ends when the number of records returned is less than `limit`.
      - May be inefficient for large offsets; the server scans+skips records each time.
    """

    # constants
    limit = query_params.get('limit', 500)
    offset = 0
    page = 0
    all_records: list = []
    timings: list = []

    # loop through pages of sized "limit" until all data fetched
    while True:
        page += 1
        params = {**query_params, 'limit': limit, 'offset': offset}

        # fetch data, get data length
        out = api_get(url, params)
        n = len(out['records'])

        # print logging
        if not quiet:
            print(f'  offset page {page:3d}  offset={offset:6d}  records={n:4d}  '
                  f'latency={out["latency_ms"]:7.1f} ms')

        # save timing info
        timings.append({
            'page':       page,
            'offset':     offset,
            'records':    n,
            'latency_ms': round(out['latency_ms'], 2),
        })
        all_records.extend(out['records'])

        if n < limit:
            break
        offset += limit

    return {'records': all_records, 'timings': pd.DataFrame(timings)}

In [ ]:
def paging_cursor(url: str, query_params: dict, quiet: bool = False) -> dict:
    """Cursor-based paging (recommended for deep / bulk pulls).
    Returns {records, timings}.

    How it works:
      - Pass `after=<cursor>` to fetch rows with _id > after.
      - On /v1/ and /v2/, the server returns an X-Next-Cursor header.
      - On /v3/, the cursor is also exposed as meta.nextCursor in the body.
      - When the cursor is absent (or meta.hasMore is false), you've reached the end.
      - Treat the cursor as opaque; some collections use string or numeric _id values.
      - Only compatible with the default sort (omit sortBy).
    """

    # constants
    limit = query_params.get('limit', 500)
    after: Optional[str] = None
    page = 0
    all_records: list = []
    timings: list = []

    # loop through pages of sized "limit" until all data fetched
    while True:
        page += 1
        # `offset` is ignored when `after` is set; strip it for clean requests.
        params = {k: v for k, v in query_params.items() if k != 'offset'}
        params['limit'] = limit
        if after is not None:
            params['after'] = after

        # fetch data, get data length
        out = api_get(url, params)
        n = len(out['records'])

        # print logging
        if not quiet:
            after_str = '(none)' if after is None else after
            print(f'  cursor page {page:3d}  after={after_str:>26s}  records={n:4d}  '
                  f'latency={out["latency_ms"]:7.1f} ms')

        # save timing info
        timings.append({
            'page':       page,
            'after':      after or '',
            'records':    n,
            'latency_ms': round(out['latency_ms'], 2),
        })
        all_records.extend(out['records'])

        # Prefer meta.nextCursor (envelope) then X-Next-Cursor header (bare).
        next_cursor: Optional[str] = None
        if out['meta'] and out['meta'].get('nextCursor'):
            next_cursor = out['meta']['nextCursor']
        else:
            # requests headers are case-insensitive; either casing works
            next_cursor = out['headers'].get('X-Next-Cursor')

        # continue to next iteration, or end
        if not next_cursor:
            break
        if n == 0:
            break
        after = next_cursor

    return {'records': all_records, 'timings': pd.DataFrame(timings)}

##
## (C) Top-level fetcher — handles version + mode selection

In [ ]:
def fetch_data(
    table: str,
    query_params: Optional[dict] = None,
    mode: str = PAGING_MODE,
    version: Optional[str] = None,
    return_as: str = 'df',
    quiet: bool = False,
):
    """Fetch every row of `table` matching `query_params`.

    Args:
        table:        path under base URL, e.g. 'stats/team/game-box' or 'teams'.
        query_params: dict of API filters.
        mode:         'cursor' (default, recommended for bulk) or 'offset'.
        version:      override the default major version for this call only.
                      None (default) -> use BASE_URL. 'v1' / 'v2' / 'v3' -> override.
        return_as:    'df' (default, DataFrame) or 'list' (raw {records, timings}).
        quiet:        suppress per-page logging.

    Returns:
        DataFrame (when return_as='df') with `df.attrs['timings']` (list of per-page dicts), OR
        dict {'records': [...], 'timings': DataFrame} when return_as='list'.
    """

    # if no query params, initialize to {}
    if query_params is None:
        query_params = {}

    # for timing
    t0 = time.perf_counter()

    # default to v3
    if version is None:
        url = f'{BASE_URL}/{table}'
    else:
        url = f'{HOST}/{version}/{table}'

    # fetch data with right mode
    if mode == 'cursor':
        result = paging_cursor(url, query_params, quiet=quiet)
    elif mode == 'offset':
        result = paging_offset(url, query_params, quiet=quiet)
    else:
        raise ValueError("mode must be 'cursor' or 'offset'")

    if return_as == 'list':
        return result

    # pd.json_normalize is robust to nested fields (flattens dot-paths)
    df = pd.json_normalize(result['records'])

    # store per-page timings as plain dicts; a DataFrame in attrs breaks print()/concat on pandas 2.2+
    df.attrs['timings'] = result['timings'].to_dict('records')

    total_ms = (time.perf_counter() - t0) * 1000.0
    if not quiet:
        print(f'  fetched {len(df)} rows x {len(df.columns)} cols from {table} in {total_ms:.1f} ms')

    return df

##
## (D) Realistic example fetches — common backfill workflows

In [ ]:
# Shared params used across the realistic fetches and the test menu below.
season_params = {'competitionIds': LATEST_COMPETITION_ID, 'splits': 'season'}

acc_params = {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID}
d1_params = {'competitionIds': LATEST_COMPETITION_ID, 'divisionIds': 1}
acc_season_params = {**acc_params, 'splits': 'season'}
acc_duke_params = {**acc_params, 'teamIds': 103549}

All use cursor pagination (recommended for bulk pulls). Use `BASE_URL`'s default version (set in section A — currently `v3`).

### (D1) Entities - competitions, conferences, teams, players

In [ ]:
all_competitions = fetch_data(
    'competitions'
)

In [ ]:
d1_conferences = fetch_data(
    'conferences',
    {'divisionIds': 1}
)

In [ ]:
d1_teams = fetch_data(
    'competition-teams',
    {'competitionIds': LATEST_COMPETITION_ID, 'divisionIds': 1}
)

In [ ]:
acc_teams = fetch_data(
    'competition-teams',
    {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID}
)

In [ ]:
d1_players = fetch_data(
    'competition-team-players',
    {'competitionIds': LATEST_COMPETITION_ID, 'divisionIds': 1}
)

In [ ]:
acc_players = fetch_data(
    'competition-team-players',
    {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID}
)

####
### (D2) Team stats for all of D1, current competition

In [ ]:
team_game_box = fetch_data(
    'stats/team/game-box',
    {'competitionIds': LATEST_COMPETITION_ID}
)

In [ ]:
team_agg_box = fetch_data(
    'stats/team/agg-box',
    {'competitionIds': LATEST_COMPETITION_ID, 'splits': 'season', 'teamOrOpponent': 'team'}
)

In [ ]:
# preview a wide stats frame (also a smoke test that printing works)
team_agg_box.head()

In [ ]:
team_agg_pbp = fetch_data(
    'stats/team/agg-pbp',
    {'competitionIds': LATEST_COMPETITION_ID, 'splits': 'season', 'teamOrOpponent': 'team'}
)

####
### (D3) Player stats — game-box can use any conference filter; agg uses splits=season

In [ ]:
acc_player_game_box = fetch_data(
    'stats/player/game-box',
    {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds':  ACC_CONFERENCE_ID}
)

In [ ]:
acc_player_game_pbp = fetch_data(
    'stats/player/game-pbp',
    {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds':  ACC_CONFERENCE_ID}
)

In [ ]:
d1_player_agg_box  = fetch_data(
    'stats/player/agg-box',
    {'competitionIds': LATEST_COMPETITION_ID, 'splits': 'season'}
)

####
### (D4) Incremental fetches via the `updated` filter (nightly sync pattern)

Server applies `updated >= <date>` (gte), and accepts `'YYYY-MM-DD'`, `'YYYY-M-D'`, or `'YYYY-MM-DDTHH:mm:ssZ'` (UTC). Use this for nightly syncs instead of full backfills — pass your last-successful-sync date as the cursor.

In [ ]:
since_date = (date.today() - timedelta(days=7)).isoformat()  # last 7 days
since_date = '2026-03-28'                                    # or pin to a fixed date

In [ ]:
recent_games = fetch_data(
    'games',
    {'competitionIds': LATEST_COMPETITION_ID, 'updated': since_date}
)

In [ ]:
recent_team_game_box = fetch_data(
    'stats/team/game-box',
    {'competitionIds': LATEST_COMPETITION_ID, 'updated': since_date}
)

In [ ]:
recent_player_game_box = fetch_data(
    'stats/player/game-box',
    {'competitionIds': LATEST_COMPETITION_ID, 'updated': since_date}
)

##
## (E) Test helpers — versatile, layer on top of `fetch_data()`

In [ ]:
def test_versions(
    table: str,
    query_params: Optional[dict] = None,
    versions: Iterable[str] = ('v1', 'v2', 'v3'),
) -> pd.DataFrame:
    """Run the same query across multiple versions; report field count / names.

    Useful for confirming v1's frozen schema is a strict subset of v2/v3, and
    for spotting newly-added v3 fields. Returns one row per version. Also
    prints the v1-vs-v3 field diff when both are in the versions list.

    Field-set inspection only needs ONE page per version — we read column
    names, not data. We call `api_get` directly (limit=1, one round-trip per
    version) instead of `fetch_data`, which would page through the full
    result set at limit=1 and look like a hang on multi-hundred-row endpoints.
    """
    if query_params is None:
        query_params = {}
    query_params = {**query_params, 'limit': query_params.get('limit', 1)}

    versions = list(versions)
    filt_str = ' '.join(f'{k}={v}' for k, v in query_params.items())
    print(f'test_versions: {table}  versions=[{", ".join(versions)}]  filters={filt_str}')

    rows = []
    for v in versions:
        url = f'{HOST}/{v}/{table}'
        print(f'  -> /{v}/{table} ...', end='', flush=True)
        out = api_get(url, query_params)

        if len(out['records']) == 0:
            df = pd.DataFrame()
        else:
            df = pd.json_normalize(out['records'])

        print(f' {len(df.columns)} fields, {len(df)} records, {out["latency_ms"]:.0f} ms')

        rows.append({
            'version':   v,
            'n_records': len(df),
            'n_fields':  len(df.columns),
            'fields':    sorted(df.columns.tolist()),
        })

    result = pd.DataFrame(rows)

    by_v = {r['version']: set(r['fields']) for r in rows}
    if 'v1' in by_v and 'v3' in by_v:
        only_in_v3 = sorted(by_v['v3'] - by_v['v1'])
        only_in_v1 = sorted(by_v['v1'] - by_v['v3'])
        delta = len(only_in_v3) - len(only_in_v1)
        print(f'\n  field diff for {table} — v1: {len(by_v["v1"])}, '
              f'v3: {len(by_v["v3"])} (delta = {delta:+d}):')
        print(f'    v3 only: {len(only_in_v3):3d}  '
              + (', '.join(only_in_v3) if only_in_v3 else '(none)'))
        print(f'    v1 only: {len(only_in_v1):3d}  '
              + (', '.join(only_in_v1) if only_in_v1 else '(none)'))
    return result

In [ ]:
def bench_pagination(
    table: str,
    query_params: Optional[dict] = None,
    modes: Iterable[str] = ('offset', 'cursor'),
    versions: Iterable[str] = ('v3',),
    quiet: bool = True,
) -> pd.DataFrame:
    """Run the same query under multiple (version × mode) combos; return timings.

    Default is /v3/ + both pagination modes — the most common A/B you actually
    care about. Pass versions=('v1','v2','v3') to compare all three.

    Wraps each combo in try/except so a single timeout/error doesn't abort
    the whole bench. Failed combos return a row with status='timeout' or
    'error' and NA timings; the loop continues to the next combo.

    Returns columns: version, mode, status, pages, records, wall_ms,
    mean_page_ms, max_page_ms.
    """
    if query_params is None:
        query_params = {}

    modes = list(modes)
    versions = list(versions)
    t0_total = time.perf_counter()

    rows = []
    for v in versions:
        for m in modes:
            print(f'--- bench: {v} + {m} on /{v}/{table} ---')
            t0_combo = time.perf_counter()
            try:
                run = fetch_data(table, query_params, mode=m, version=v,
                                 return_as='list', quiet=quiet)
                elapsed_combo_ms = (time.perf_counter() - t0_combo) * 1000.0
                t = run['timings']
                rows.append({
                    'version':      v,
                    'mode':         m,
                    'status':       'ok',
                    'pages':        len(t),
                    'records':      len(run['records']),
                    'wall_ms':      round(float(t['latency_ms'].sum()), 1),
                    'mean_page_ms': round(float(t['latency_ms'].mean()), 1),
                    'max_page_ms':  round(float(t['latency_ms'].max()), 1),
                })
            except Exception as e:
                elapsed_combo_ms = (time.perf_counter() - t0_combo) * 1000.0
                err_msg = str(e)
                status = 'timeout' if re.search(r'timeout|timed? out', err_msg, re.IGNORECASE) else 'error'
                print(f'  -> {status} after {elapsed_combo_ms / 1000:.1f}s: {err_msg}')
                rows.append({
                    'version':      v,
                    'mode':         m,
                    'status':       status,
                    'pages':        pd.NA,
                    'records':      pd.NA,
                    'wall_ms':      round(elapsed_combo_ms, 1),
                    'mean_page_ms': pd.NA,
                    'max_page_ms':  pd.NA,
                })

    result = pd.DataFrame(rows)
    total_elapsed = time.perf_counter() - t0_total

    # --- Summary block ---
    print(f'\nbench_pagination({table}) — total wall {total_elapsed:.1f}s:')
    print(result.to_string(index=False))

    # Headline cursor-vs-offset comparison, one line per version (only when both modes ran).
    # If either side timed out or errored, we surface that instead of a misleading ratio.
    def fmt_side(row: pd.Series) -> str:
        if row.empty:
            return '(missing)'
        if row['status'] != 'ok':
            return f'DNF [{row["status"]} after {row["wall_ms"] / 1000:.1f}s]'
        return f'{row["wall_ms"] / 1000:.1f}s (mean {row["mean_page_ms"]:.0f} / max {row["max_page_ms"]:.0f} ms)'

    if 'offset' in modes and 'cursor' in modes:
        for v in versions:
            off = result[(result['version'] == v) & (result['mode'] == 'offset')]
            cur = result[(result['version'] == v) & (result['mode'] == 'cursor')]
            off_row = off.iloc[0] if len(off) else pd.Series(dtype=object)
            cur_row = cur.iloc[0] if len(cur) else pd.Series(dtype=object)
            both_ok = (
                not off_row.empty and not cur_row.empty
                and off_row['status'] == 'ok' and cur_row['status'] == 'ok'
            )
            if both_ok:
                ratio = off_row['max_page_ms'] / cur_row['max_page_ms']
                print(f'  /{v}/  offset {fmt_side(off_row)}  vs  cursor {fmt_side(cur_row)}  '
                      f'|  max-page ratio {ratio:.1f}x')
            else:
                print(f'  /{v}/  offset {fmt_side(off_row)}  vs  cursor {fmt_side(cur_row)}')
    print()

    return result

##
## (F) Speed Tests — exercise the helpers above

### Test 1: field-set comparison across v1, v2, v3

Confirms v1's frozen schema is a strict subset of v2/v3, and surfaces fields that have been added to the underlying collection since v1 was frozen.

In [ ]:
test_versions(
    'stats/team/game-box',
    {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID}
)

In [ ]:
test_versions(
    'stats/player/game-pbp',
    {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID}
)

In [ ]:
test_versions(
    'stats/player/agg-pbp',
    {'competitionIds': LATEST_COMPETITION_ID, 'conferenceIds': ACC_CONFERENCE_ID}
)

In [ ]:
test_versions(
    'competition-teams',
    {'divisionIds': 1}
)

In [ ]:
test_versions(
    'competition-team-players',
    {'divisionIds': 1}
)

####
### Test 2: cursor vs offset on a small endpoint, default version (v3)

Quick sanity check; both modes should return the same record count.

In [ ]:
bench_pagination(
    'games',
    {'competitionIds': LATEST_COMPETITION_ID, 'limit': 500},
    quiet=False,
)

####
### Test 3: cursor vs offset on `competition-team-players`

`competition-team-players` for a full competition is ~25k rows — where offset pagination starts to slow down (database scans + skips) and cursor stays flat. Watch `max_page_ms`: offset's max should grow with depth, cursor's stays roughly constant.

In [ ]:
bench_pagination(
    'competition-team-players',
    {'competitionIds': LATEST_COMPETITION_ID, 'limit': 1000}
)

####
### Test 4: cursor vs offset on a bigger endpoint

At `limit=1000`, this should return ~140 pages to get ~140K player-game stats for a competition. Deep-pagination is where the offset/cursor gap is most obvious.

In [ ]:
bench_pagination(
    'stats/player/game-box',
    {'competitionIds': LATEST_COMPETITION_ID, 'limit': 1000},
    quiet=False
)

####
### Test 5: envelope shape demo

`/v3/` returns `{ response: { meta, data } }`; `api_get` unwraps `records` and exposes `meta`. `/v1/` and `/v2/` return bare arrays; `meta` is `None`.

In [ ]:
v3_page = api_get(
    f'{HOST}/v3/games',
    {'competitionIds': LATEST_COMPETITION_ID, 'limit': 3},
)
print('v3 envelope meta:')
print(v3_page['meta'])

In [ ]:
v2_page = api_get(
    f'{HOST}/v2/games',
    {'competitionIds': LATEST_COMPETITION_ID, 'limit': 3},
)
print('v2 bare-array meta (None expected):')
print(v2_page['meta'])

####
### Test 6: incremental fetch via `updated` — cursor vs offset benchmark

The `updated` filter is the common pattern for incremental ingestion: fetch only records modified on or after a given UTC date. Use this for nightly syncs instead of full backfills.

- **Server format**: `'YYYY-MM-DD'`, `'YYYY-M-D'`, or `'YYYY-MM-DDTHH:mm:ssZ'` (UTC).
- **Server applies** `updated >= <value>` (gte). Pass your last-successful-sync date as the cursor.
- **For tiny windows** (one day) where everything fits in one page, both modes are equivalent. For bigger windows that span many pages, cursor pulls ahead.

In [ ]:
# set params
since = '2026-03-28'   # before March Madness
recent_params = {'competitionIds': LATEST_COMPETITION_ID, 'updated': since}

In [ ]:
incremental_endpoints = ['games', 'stats/team/game-box', 'stats/player/game-box']

# bench_pagination prints per-endpoint cursor-vs-offset comparison and returns the per-(mode, version) DataFrame.
# assign(table=...) tags each row with its endpoint so the cross-endpoint summary at the bottom is readable.
incremental_summary = pd.concat([
    bench_pagination(table, recent_params).assign(table=table)
    for table in incremental_endpoints
], ignore_index=True)

# Reorder columns to put `table` first
incremental_summary = incremental_summary[
    ['table'] + [c for c in incremental_summary.columns if c != 'table']
]

In [ ]:
print('\nTest 6 cross-endpoint summary (each endpoint x each mode):')
print(incremental_summary.to_string(index=False))